Q1: What does the Embedding layer do to a word index?

It converts word indices into dense numerical vectors that represent words.

Q2: What is the role of the Encoder LSTM?

It reads the input sentence and learns its meaning and context.

Q3: What does the Decoder LSTM do?

It uses the encoded information to generate the translated sentence.

Q4: Why do we use softmax in the final layer?

It converts the output into probabilities to predict the most likely word.

In [1]:

from google.colab import files

uploaded = files.upload()


Saving archive (38).zip to archive (38).zip


In [2]:

import zipfile
import os

zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

for root, dirs, files_list in os.walk('/content/dataset'):
    for file in files_list:
        print(os.path.join(root, file))


/content/dataset/eng_-french.csv


In [3]:

import pandas as pd
import glob

csv_files = glob.glob('/content/dataset/**/*.csv', recursive=True)

print("CSV files:", csv_files)

if csv_files:
    df = pd.read_csv(csv_files[0])
    print(df.head())
    print(df.columns.tolist())
else:
    print("No CSV file found. Check the extracted files.")


CSV files: ['/content/dataset/eng_-french.csv']
  English words/sentences French words/sentences
0                     Hi.                 Salut!
1                    Run!                Cours !
2                    Run!               Courez !
3                    Who?                  Qui ?
4                    Wow!             Ça alors !
['English words/sentences', 'French words/sentences']


In [4]:

import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split


In [6]:

english_col = df.columns[0]
french_col = df.columns[1]

df = df[[english_col, french_col]].dropna()
df = df.astype(str)

# Use a subset for faster training
df = df.head(5000)

english_sentences = df[english_col].tolist()
french_sentences = df[french_col].tolist()

tokenizer_eng = Tokenizer(oov_token="<OOV>")
tokenizer_frn = Tokenizer(oov_token="<OOV>")

tokenizer_eng.fit_on_texts(english_sentences)
tokenizer_frn.fit_on_texts(french_sentences)

eng_sequences = tokenizer_eng.texts_to_sequences(english_sentences)
frn_sequences = tokenizer_frn.texts_to_sequences(french_sentences)

eng_padded = pad_sequences(eng_sequences)
frn_padded = pad_sequences(frn_sequences)

print("English shape:", eng_padded.shape)
print("French shape:", frn_padded.shape)


English shape: (5000, 4)
French shape: (5000, 10)


In [7]:

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(eng_padded.shape[1],)),
    tf.keras.layers.Embedding(
        input_dim=len(tokenizer_eng.word_index) + 1,
        output_dim=64
    ),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.RepeatVector(frn_padded.shape[1]),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(
            len(tokenizer_frn.word_index) + 1,
            activation="softmax"
        )
    )
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 4, 64)          │        79,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 10, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 10, 2723)       │       176,995 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 322,851 (1.23 MB)

 Trainable params: 322,851 (1.23 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:

eng_train, eng_test, frn_train, frn_test = train_test_split(
    eng_padded,
    frn_padded,
    test_size=0.2,
    random_state=42
)

history = model.fit(
    eng_train,
    np.expand_dims(frn_train, -1),
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

loss, accuracy = model.evaluate(
    eng_test,
    np.expand_dims(frn_test, -1)
)

print("Test Accuracy:", accuracy)


Epoch 1/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - accuracy: 0.6829 - loss: 4.3876 - val_accuracy: 0.6865 - val_loss: 2.7035
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.6903 - loss: 2.6099 - val_accuracy: 0.6865 - val_loss: 2.7185
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - accuracy: 0.6903 - loss: 2.5987 - val_accuracy: 0.6865 - val_loss: 2.7385
Epoch 4/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.6903 - loss: 2.5939 - val_accuracy: 0.6865 - val_loss: 2.7545
Epoch 5/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.6903 - loss: 2.5918 - val_accuracy: 0.6865 - val_loss: 2.7703
Epoch 6/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - accuracy: 0.6903 - loss: 2.5906 - val_accuracy: 0.6865 - val_loss: 2.7822
Epoch 7/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 35ms/step - accuracy: 0.6903 - loss: 2.5901 - val_accuracy: 0.6865 - val_loss: 2.7935
Epoch 8/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.6903 - loss: 2.5890 - val_accu

In [9]:

def translate_sentence(sentence):
    sequence = tokenizer_eng.texts_to_sequences([sentence])

    padded = pad_sequences(
        sequence,
        maxlen=eng_padded.shape[1]
    )

    prediction = model.predict(padded, verbose=0)

    predicted_ids = np.argmax(prediction, axis=-1)

    translation = tokenizer_frn.sequences_to_texts(
        predicted_ids
    )[0]

    return translation

sentence = "Hello"
print("English:", sentence)
print("French:", translate_sentence(sentence))


English: Hello
French: <OOV> <OOV> <OOV> <OOV> <OOV> <OOV> <OOV> <OOV> <OOV> <OOV>
